# Detection Pipeline (PKU-MMD / TSU)
Pipeline trich xuat bounding box nguoi bang YOLO26. Chay tuan tu cac cell.

In [1]:
# # Cell 1: Lay code moi nhat tu GitHub (nhanh dai)
# import os
# REPO = "https://github.com/tuan8p/Skeleton-EAA-Pose.git"
# BRANCH = "dai"
# WORKDIR = "/content/Skeleton-EAA-Pose"
# if not os.path.isdir(WORKDIR):
#     !git clone -b {BRANCH} {REPO} {WORKDIR}
# else:
#     !git -C {WORKDIR} pull origin {BRANCH}
# %cd {WORKDIR}
%cd "D:\Downloads\ĐATN\Skeleton-EAA-Pose"

D:\Downloads\ĐATN\Skeleton-EAA-Pose


In [2]:
# Cell 2: Cai dat thu vien
# !pip install -q mediapipe opencv-python numpy scipy pyyaml ultralytics tqdm psutil
!pip install -r requirements.txt

In [3]:
# # Cell 3: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
# Cell 4: Cau hinh - chon dataset va chinh path
DATASET = "PKU"   # "PKU" hoac "TSU"

# PKU_PATHS = {
#     "video_dir": '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets/PKUMMD/Data/RGB_VIDEO',
#     "annotation_dir": '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets/skeletons/PKU/Label_PKUMMD_v1',
# }
# TSU_PATHS = {
#     "video_dir": '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets/videos/TSU',
#     "annotation_dir": '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets/skeletons/TSU/Annotation_v1.0',
# }
DEPTH_PATHS = {   # PKU: <video>-infrared.avi ; TSU: <video>-depth.mp4
    "PKU": "/content/drive/MyDrive/PKU-MMD/DEPTH_PKUMMD",
    "TSU": "/content/drive/MyDrive/TSU/videos",
}
# OUTPUT_PATHS = {
#     "output_dir": '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets/outputs_detection',
# }

PKU_PATHS = {
    "video_dir": r'D:\Downloads\ĐATN\RGB_PKUMMDv1',
    "annotation_dir": r'D:\Downloads\ĐATN\PKU-MMD\Label_PKUMMD',
}
TSU_PATHS = {
    "video_dir": '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets/videos/TSU',
    "annotation_dir": '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets/skeletons/TSU/Annotation_v1.0',
}
OUTPUT_PATHS = {
    "output_dir": r'D:\Downloads\ĐATN',
}

# ==========================================
# CAU HINH PIPELINE (Chinh truc tiep o day)
# ==========================================
SETTINGS = {
    "yolo.model_name": "yolo26x.pt",       # Chon model: yolo26n.pt (nhe), yolo26l.pt, yolo26x.pt (nang)
    "yolo.tracker": "iou",               # Thay deepocsort thanh iou de tranh mat frame
    "runtime.num_workers": 2,            # CHAY SONG SONG 4 VIDEO (Can tat TTA de tranh tran VRAM)
    "yolo.batch_size": 32,               # Tang len 16/32 de tan dung 15GB VRAM cua T4
    "yolo.conf_threshold": 0.2,          # Nguong tu tin de loai box rac
    "yolo.iou_threshold": 0.7,           # Nguong IoU
    "detection.retry.max_retries": 2,    # Retry 1 (1280+TTA), Retry 2 (CLAHE)
    "detection.retry.imgsz_retry": 1280, # Zoom in anh
    "detection.retry.tta": False,         # Test Time Augmentation
    "detection.retry.clahe_on_retry": True,
    "chunk.max_actions_per_chunk": 5,    # Tranh load RAM qua to
    # --- TSU Specific Config ---
    "tsu.video_ext": ".mp4",
    "tsu.has_header": True,
    "tsu.event_column": 0,
    "tsu.start_column": 1,
    "tsu.end_column": 2,
    "tsu.event_map": {},          # {"event_name": action_id}
    "tsu.event_mapping_path": "event_mapping.csv", # File anh xa event TSU co dinh
}

USE_DEPTH = False   # True = bat ghost legs masking; "camera" 3D khuyen nghi chi cho PKU
START, END = 0, 1   # chi so video trong danh sach annotation [start, end)


In [5]:
# # Cell 5: Tai model BlazePose (Tasks API) neu chua co
# import os, urllib.request
# os.makedirs("models", exist_ok=True)
# MODEL = "models/pose_landmarker_full.task"
# if not os.path.exists(MODEL):
#     urllib.request.urlretrieve(
#         "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
#         MODEL)
# print("model ready:", os.path.getsize(MODEL), "bytes")

In [ ]:
# Cell 6: Nap config, ghi de path theo dataset da chon
from src.config_manager import ConfigManager
cfg = ConfigManager("config.yaml")
cfg.set("dataset", DATASET)
for k, v in (PKU_PATHS if DATASET == "PKU" else TSU_PATHS).items():
    cfg.set(f"paths.{k}", v)
cfg.set("detection.output_dir", OUTPUT_PATHS["output_dir"])
cfg.set("paths.depth_dir", DEPTH_PATHS[DATASET])
cfg.set("depth.enabled", USE_DEPTH)

# Ghi de cac cau hinh Pipeline (tu Cell 4)
for k, v in SETTINGS.items():
    cfg.set(k, v)

cfg.save("config.runtime.yaml")   # luu lai de kiem tra/reproduce
print(cfg.as_dict())


{'dataset': 'PKU', 'mediapipe': {'model_path': 'models/pose_landmarker_full.task', 'num_poses': 1, 'min_detection_confidence': 0.5, 'min_presence_confidence': 0.5, 'min_tracking_confidence': 0.5, 'static_image_mode': False}, 'yolo': {'model_name': 'yolo26x.pt', 'batch_size': 32, 'conf_threshold': 0.2, 'iou_threshold': 0.7, 'max_detections': 2, 'tracker': 'iou'}, 'depth': {'enabled': False, 'gray_to_m': 0.256, 'scale_to_rgb': True, 'masking': {'enabled': True, 'min_delta': 15, 'fill_color': [128, 128, 128]}, 'intrinsics': {'fx': 1059.29, 'fy': 1059.32, 'cx': 962.9, 'cy': 543.4}}, 'preprocessing': {'resize': {'enabled': True, 'width': 640, 'height': 640, 'keep_ratio': True}, 'clahe': {'enabled': True, 'clip_limit': 2.0, 'tile_grid_size': 8}, 'gamma': {'enabled': True, 'value': 0.8}, 'denoise': {'enabled': True, 'method': 'bilateral', 'strength': 5}}, 'thresholds': {'confidence': 0.2, 'max_nan_ratio': 0.45}, 'validator': {'max_retries': 2}, 'interpolation': {'max_gap': 3}, 'temporal': {'e

: 

In [ ]:
# Cell 7: Chay pipeline (co resume neu chay lai)
from src.detection_pipeline import DetectionPipeline
orch = DetectionPipeline(cfg=cfg)
orch.run_batch(start=START, end=END)

Videos (Parallel):   0%|          | 0/1 [00:00<?, ?it/s]